# Murphy 1 · Chapter 15 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M15). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 15.1 · Scaled dot-product attention & causal masking from scratch  
*Section §15.4 · code*

Given a 4-token sequence with d_k = 4 query/key vectors Q, K and value vectors V, compute (a) bidirectional scaled dot-product attention weights and output, and (b) causal (autoregressive) attention where token t cannot attend to future positions j > t. Verify that every row of both attention matrices sums to 1.

<details><summary><b>Hint</b></summary>

Scale Q @ K.T by 1/√d_k; for causal attention, add an upper-triangular mask of −1e9 (above the diagonal) before softmax.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(42)
T, d = 4, 4
Q = rng.normal(size=(T, d)); K = rng.normal(size=(T, d)); V = rng.normal(size=(T, d))
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
scores = ...  # TODO: scaled dot-product similarity
A_bi = softmax(scores)
mask = ...  # TODO: block future positions j > i
A_causal = softmax(scores + mask)
print("bidirectional weights:\n", A_bi.round(3))
print("causal weights:\n", A_causal.round(3))
print("row sums:", A_bi.sum(axis=1).round(6), A_causal.sum(axis=1).round(6))

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(42)
T, d = 4, 4
Q = rng.normal(size=(T, d)); K = rng.normal(size=(T, d)); V = rng.normal(size=(T, d))
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
scores = (Q @ K.T) / np.sqrt(d)  # scaled dot-product similarity
A_bi = softmax(scores)
mask = np.triu(np.full((T, T), -1e9), k=1)  # block future positions j > i
A_causal = softmax(scores + mask)
print("bidirectional weights:\n", A_bi.round(3))
print("causal weights:\n", A_causal.round(3))
print("row sums:", A_bi.sum(axis=1).round(6), A_causal.sum(axis=1).round(6))

<details><summary><b>Answer and explanation</b></summary>

In bidirectional attention every token blends all 4 positions; in causal attention the upper triangle of the weight matrix is strictly 0.0, so token 0 only sees token 0 (weight 1.0) while token 3 blends all four (Murphy §15.4–15.5, D2L §11.3).

Expected output:
```
bidirectional weights:
 [[0.311 0.268 0.215 0.206]
 [0.234 0.348 0.314 0.104]
 [0.276 0.275 0.217 0.232]
 [0.147 0.192 0.156 0.505]]
causal weights:
 [[1.    0.    0.    0.   ]
 [0.402 0.598 0.    0.   ]
 [0.36  0.358 0.282 0.   ]
 [0.147 0.192 0.156 0.505]]
row sums: [1. 1. 1. 1.] [1. 1. 1. 1.]
```

</details>

## 15.2 · Why dividing by √d keeps softmax gradients alive  
*Section §15.4 · derive*

Suppose q and k have independent entries with mean 0 and variance 1. Find the mean and variance of q·k in dimension d, and explain what happens to the softmax Jacobian ∂softmax(z)/∂z when d = 512 without the 1/√d scale.

<details><summary><b>Hint</b></summary>

Var(Σᵢ qᵢ kᵢ) = Σᵢ Var(qᵢ kᵢ), and for independent mean-zero variables Var(qᵢ kᵢ) = Var(qᵢ)Var(kᵢ) = 1.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

Each product qᵢkᵢ has mean 0 and variance 1, so the sum of d independent terms q·k has mean 0 and variance d (standard deviation √d). At d = 512, typical dot products are ±22.6; feeding ±22.6 into exp() makes the largest score take probability α_max ≈ 1 − 10⁻¹⁰ and all others ≈ 0, where the softmax gradient αᵢ(1 − αᵢ) vanishes. Dividing by √d restores unit variance regardless of d (Murphy §15.4.2).

</details>